In [1]:
import numpy as np
import tensorflow as tf

from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split
from sklearn.linear_model import SGDClassifier
from sklearn.preprocessing import StandardScaler
from dataloader import load_shards

# Use your local filepath here
file_pattern = r'G:/.shortcut-targets-by-id/1abX3CWvYUSJM3cGg6r_GeHAVeNoYH6Ul/CS6140_Project_Data/koppen_shard_part_*.tfrecord.gz'
all_files = tf.io.gfile.glob(file_pattern)

In [3]:
# Test the loader
loaded_shards = load_shards(all_files)
labels = set()
for batch_tensors, batch_labels in loaded_shards.take(1).as_numpy_iterator():
    print(batch_tensors.shape)
    print(batch_labels.shape)

(32, 129, 129, 12)
(32,)


In [4]:
train_files, test_files = train_test_split(all_files, test_size=0.2, random_state=42)
print(f"Training on {len(train_files)} shards.")
print(f"Testing on {len(test_files)} shards.")

Training on 1200 shards.
Testing on 300 shards.


In [30]:
scaler = StandardScaler()
all_classes = np.arange(1, 31)
batch_size = 128
train_dataset = load_shards(train_files).batch(batch_size)

# --- Pass 1: Calculating scaling statistics ---
print("Pass 1: Calculating scaling statistics...")
for X_batch, _ in train_dataset.as_numpy_iterator():
    X_flat = X_batch.reshape(X_batch.shape[0], -1)
    scaler.partial_fit(X_flat)

print("Scaling statistics computed")

Pass 1: Calculating scaling statistics...
Scaling statistics computed


In [40]:
# Use SGDClassifier with loss='log_loss' to implement Logistic Regression with partial_fit
clf = SGDClassifier(
    loss='log_loss',
    penalty='l2',
    alpha=10,
    random_state=42
)

# --- Pass 2: Model training ---
print('Pass 2: Training model...')
for i, (X_batch, y_batch) in enumerate(train_dataset.as_numpy_iterator()):
    X_flat = X_batch.reshape(X_batch.shape[0], -1)

    # Apply scaling
    X_scaled = scaler.transform(X_flat)

    # Fit model on current batch
    clf.partial_fit(X_scaled, y_batch, classes=all_classes)

    if i % 20 == 0:
        print(f"Trained on batch {i}")

print("Training completed")

Pass 2: Training model...
Trained on batch 0
Trained on batch 20
Trained on batch 40
Trained on batch 60
Trained on batch 80
Trained on batch 100
Trained on batch 120
Trained on batch 140
Trained on batch 160
Trained on batch 180
Trained on batch 200
Trained on batch 220
Trained on batch 240
Trained on batch 260
Trained on batch 280
Trained on batch 300
Trained on batch 320
Trained on batch 340
Trained on batch 360
Training completed


In [41]:
# --- Evaluation ---
print("Evaluating on training set...")
y_train_true, y_train_pred = [], []

for X_batch, y_batch in train_dataset.as_numpy_iterator():
    X_flat = X_batch.reshape(X_batch.shape[0], -1)
    X_scaled = scaler.transform(X_flat)

    y_train_pred.extend(clf.predict(X_scaled))
    y_train_true.extend(y_batch)

print("Evaluating on test set...")
test_ds = load_shards(test_files).batch(batch_size)
y_test_true, y_test_pred = [], []

for X_batch, y_batch in test_ds.as_numpy_iterator():
    X_flat = X_batch.reshape(X_batch.shape[0], -1)
    X_scaled = scaler.transform(X_flat)

    y_test_pred.extend(clf.predict(X_scaled))
    y_test_true.extend(y_batch)

Evaluating on training set...
Evaluating on test set...


In [42]:
print("Training classification report:")
print(classification_report(y_train_true, y_train_pred, zero_division=0))
print('\n\n')
print("Test classification report:")
print(classification_report(y_test_true, y_test_pred, zero_division=0))

Training classification report:
              precision    recall  f1-score   support

           1       0.53      0.04      0.07      1275
           2       0.27      0.70      0.39      1557
           3       0.27      0.17      0.21      1542
           4       0.65      0.36      0.47      1571
           5       0.21      0.55      0.30      1639
           6       0.25      0.00      0.00      1558
           7       0.20      0.01      0.02      1601
           8       0.14      0.23      0.18      1633
           9       0.19      0.03      0.05      1597
          10       0.00      0.00      0.00      1604
          11       0.36      0.01      0.01      1615
          12       0.00      0.00      0.00      1601
          13       0.75      0.37      0.50      1559
          14       0.35      0.00      0.01      1602
          15       0.19      0.35      0.25      1453
          16       0.11      0.01      0.02      1599
          17       0.22      0.67      0.34      